# 🤖 Tiny GPT from Scratch

**Goal:** build and train your own small **GPT** — the same kind of model as ChatGPT, just much smaller — from an empty file. It will learn to write Shakespeare-style text one character at a time. Then you can train it on **Hindi or Marathi** text too.

This notebook follows the ideas in Andrej Karpathy's excellent video **["Let's build GPT: from scratch, in code, spelled out"](https://www.youtube.com/watch?v=kCc8FmEb1nY)**. Watch it alongside this notebook — pause the video, run the matching section here.

**New skills in this project**
- **Language modelling** — predicting the next token
- **Tokenisation** (characters → numbers)
- **Self-attention**: queries, keys, values, and the causal mask
- The **Transformer block**: multi-head attention, feed-forward, residual connections, layer norm
- Training a model with ~millions of parameters and **sampling** from it (temperature)

⏱️ A month or more if you do it properly (watch the whole video series). **How to run:** open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file. Run cells top to bottom with **Shift + Enter**.

⚡ **Turn on the free GPU first:** in Colab, **Runtime → Change runtime type → T4 GPU → Save**. Training will be 10–50× faster.


**Before this project:** make sure you're comfortable with the digit recognizer (PyTorch training loop) and movie recommender (embeddings).

## 1. Setup

In [ ]:
import os, math, time, urllib.request
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337)
print("using", device)

## 2. The data

By default: **Tiny Shakespeare** — 1 MB of Shakespeare's plays.

🇮🇳 **Want Hindi or Marathi?** Put any large `.txt` file (at least ~1 MB — e.g. stories, news articles or Wikipedia text, saved as UTF-8) in Colab's Files panel and set `TEXT_FILE` to its name. Everything else works unchanged, because we model individual characters — Devanagari included.

In [ ]:
TEXT_FILE = "input.txt"      # ✏️ change to e.g. "marathi.txt" to use your own text
if TEXT_FILE == "input.txt" and not os.path.exists(TEXT_FILE):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", TEXT_FILE)

with open(TEXT_FILE, encoding="utf-8") as f:
    text = f.read()
print(f"{len(text):,} characters")
print(text[:500])

## 3. Tokenisation — turning text into numbers

A model needs numbers. The simplest **tokeniser**: give every distinct character an ID. (ChatGPT uses *sub-word* tokens — chunks like `"ing"` or `" the"` — but the idea is identical.)

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print("vocabulary:", "".join(chars))
print("vocab size:", vocab_size)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print(encode("hello there"))
print(decode(encode("hello there")))

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]      # first 90% to train, last 10% to validate
print(train_data.shape, val_data.shape)

## 4. The task: predict the next character

We show the model a chunk of text and at **every position** it must predict the next character. One chunk of 8 characters gives 8 training examples:

In [ ]:
block_size = 8
x, y = train_data[:block_size], train_data[1:block_size + 1]
for t in range(block_size):
    print(f"given {decode(x[:t + 1].tolist())!r:14s} → predict {decode([y[t].item()])!r}")

In [ ]:
def get_batch(split, batch_size, block_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", 4, 8)
print("inputs:\n", xb, "\ntargets:\n", yb)

## 5. The simplest language model: bigram

A **bigram** model predicts the next character from **only the current one** — a lookup table of "after `q`, `u` is likely". It's our baseline.

In [ ]:
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)   # row = current char, values = scores for next char

    def forward(self, idx, targets=None):
        logits = self.table(idx)                               # (batch, time, vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

bigram = BigramLM(vocab_size).to(device)
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print("Untrained bigram says:\n", decode(bigram.generate(start, 200)[0].tolist()))

In [ ]:
@torch.no_grad()
def estimate_loss(model, block_size, batch_size=32, iters=50):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(iters)
        for k in range(iters):
            X, Y = get_batch(split, batch_size, block_size)
            losses[k] = model(X, Y)[1].item()
        out[split] = losses.mean().item()
    model.train()
    return out

opt = torch.optim.AdamW(bigram.parameters(), lr=1e-2)
for step in range(3000):
    xb, yb = get_batch("train", 32, 8)
    _, loss = bigram(xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
print("bigram loss:", estimate_loss(bigram, 8))
print(f"(random guessing would be ln({vocab_size}) = {math.log(vocab_size):.2f})\n")
print(decode(bigram.generate(start, 300)[0].tolist()))

It learned which letters follow which, but it has **no memory** of anything earlier. To write real words and sentences, each position must be able to *look back* at the previous characters. That's what **attention** does.

## 6. Self-attention, step by step

### 6a. The trick: averaging the past with a matrix multiply
Simplest way to "look back": each position takes the **average of all previous positions**. A lower-triangular matrix does this in one step — and crucially it **never looks at the future** (that would be cheating: the future is what we're predicting).

In [ ]:
T = 5
tril = torch.tril(torch.ones(T, T))
weights = tril / tril.sum(1, keepdim=True)
print("averaging weights (row t = how much position t looks at each earlier position):")
print(weights.round(decimals=2))

### 6b. Letting the data decide the weights: queries, keys and values

Plain averaging treats all past characters equally. **Self-attention** lets each position decide what's relevant:

- Every token emits a **query** — *"what am I looking for?"*
- Every token emits a **key** — *"what do I contain?"*
- Affinity = query · key. High where they match.
- Every token also emits a **value** — *"what I'll pass on if you attend to me"*.

Then: **mask the future**, apply softmax to get weights, and take the weighted sum of values.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

(This exact formula is on page 4 of the paper from project 8!)

In [ ]:
torch.manual_seed(0)
B, T, C, head_size = 1, 6, 16, 8
x = torch.randn(B, T, C)                       # 6 tokens, each a 16-number vector
query, key, value = nn.Linear(C, head_size, bias=False), nn.Linear(C, head_size, bias=False), nn.Linear(C, head_size, bias=False)
q, k, v = query(x), key(x), value(x)
scores = q @ k.transpose(-2, -1) / head_size ** 0.5     # (B, T, T) affinities
scores = scores.masked_fill(torch.tril(torch.ones(T, T)) == 0, float("-inf"))   # no peeking at the future
att = F.softmax(scores, dim=-1)
out = att @ v
print("attention weights:\n", att[0].detach().round(decimals=2))
print("output shape:", out.shape)

## 7. Building the full GPT

A GPT is a stack of **Transformer blocks**. Each block has:
1. **Multi-head attention** — several attention heads in parallel, each can look for different things (one for vowels, one for the speaker's name…).
2. **Feed-forward network** — each position "thinks" about what it gathered.
3. **Residual connections** (`x + ...`) and **LayerNorm** — make deep networks trainable.

We also add **position embeddings**, since attention alone doesn't know word order.

```
tokens → token embedding + position embedding
       → [Block] × n_layer      (Block = LayerNorm → Multi-head attention → + ; LayerNorm → FeedForward → +)
       → LayerNorm → Linear → scores for the next character
```

In [ ]:
class Head(nn.Module):
    """One head of causal self-attention."""
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        w = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        w = w.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        w = self.dropout(F.softmax(w, dim=-1))
        self.last_attention = w.detach()          # saved so we can visualise it later
        return w @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
                                 nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # communicate: tokens look at each other
        x = x + self.ffwd(self.ln2(x))     # compute: each token thinks on its own
        return x


class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        logits = self.head(self.ln_f(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])        # only the last block_size tokens fit
            logits = logits[:, -1, :] / temperature
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
        self.train()
        return idx

## 8. Train it

The settings below pick a model size for your hardware. On a Colab **T4 GPU**, the default takes ~10 minutes. Set `BIG = True` for Karpathy's full-size model (~10 M parameters, ~30–45 minutes on a T4, noticeably better text).

In [ ]:
BIG = False
if device == "cuda":
    cfg = dict(n_embd=384, n_head=6, n_layer=6, block_size=256, dropout=0.2, batch_size=64, max_iters=5000, lr=3e-4) if BIG \
        else dict(n_embd=256, n_head=4, n_layer=4, block_size=128, dropout=0.2, batch_size=64, max_iters=3000, lr=1e-3)
else:   # CPU: small and quick, so you can still see it learn
    cfg = dict(n_embd=128, n_head=4, n_layer=3, block_size=64, dropout=0.1, batch_size=32, max_iters=1500, lr=1e-3)
print(cfg)

model = GPT(vocab_size, cfg["n_embd"], cfg["n_head"], cfg["n_layer"], cfg["block_size"], cfg["dropout"]).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["max_iters"])
eval_every = max(1, cfg["max_iters"] // 10)
log = []
t0 = time.time()
for it in range(cfg["max_iters"] + 1):
    if it % eval_every == 0:
        losses = estimate_loss(model, cfg["block_size"], cfg["batch_size"], iters=30)
        log.append((it, losses["train"], losses["val"]))
        print(f"step {it:5d} | train loss {losses['train']:.3f} | val loss {losses['val']:.3f} | {time.time() - t0:.0f}s")
    if it == cfg["max_iters"]:
        break
    xb, yb = get_batch("train", cfg["batch_size"], cfg["block_size"])
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step(); scheduler.step()

In [ ]:
steps, tr, va = zip(*log)
plt.figure(figsize=(8, 4))
plt.plot(steps, tr, label="train"); plt.plot(steps, va, label="val")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3); plt.title("Tiny GPT training")
plt.show()

## 9. ✍️ Let it write

In [ ]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)   # start from a newline
print(decode(model.generate(start, 600, temperature=0.8)[0].tolist()))

### Temperature
The model outputs probabilities for the next character; we **sample** from them.
- **Low temperature (0.5)** → safer, more repetitive.
- **High temperature (1.3)** → more creative, more mistakes.

You can also give it a **prompt** to continue:

In [ ]:
prompt = "ROMEO:\n"     # ✏️ try your own (only characters that appear in the training text)
ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
for temp in [0.5, 1.0, 1.3]:
    print(f"--- temperature {temp} ---")
    print(decode(model.generate(ids, 250, temperature=temp, top_k=20)[0].tolist()), "\n")

## 10. Peek inside: what does attention look at?
Each row shows, for one character, how strongly it attends to each earlier character (brighter = more).

In [ ]:
sample = "First Citizen:\nBefore we proceed any further, hear me speak."
sample = "".join(c for c in sample if c in stoi)[:cfg["block_size"]] or text[:60]
ids = torch.tensor([encode(sample)], device=device)
model.eval()
with torch.no_grad():
    model(ids)
model.train()
layer, n_show = len(model.blocks) - 1, min(4, len(model.blocks[0].attn.heads))
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 5))
labels = [c if c != "\n" else "⏎" for c in sample]
for h, ax in enumerate(axes if n_show > 1 else [axes]):
    w = model.blocks[layer].attn.heads[h].last_attention[0].cpu()
    ax.imshow(w, cmap="viridis")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=6)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=6)
    ax.set_title(f"layer {layer + 1}, head {h + 1}")
plt.tight_layout(); plt.show()

The black upper-right triangle is the **causal mask** — no position can look at the future.

In [ ]:
torch.save({"model": model.state_dict(), "cfg": cfg, "chars": chars}, "tiny_gpt.pt")
print("Saved tiny_gpt.pt")

## 🎓 How this relates to ChatGPT

| | Your Tiny GPT | GPT-3 |
|---|---|---|
| Parameters | ~1–10 million | 175 **billion** |
| Tokens | characters (~65) | sub-words (~50,000) |
| Context | 128–256 characters | 2,048 tokens |
| Training text | 1 MB | ~570 GB |

Same architecture, same training objective (predict the next token). ChatGPT then adds **fine-tuning on conversations** and **reinforcement learning from human feedback** to turn a text-continuer into an assistant.

## ✅ What you learned
- Language models are trained to **predict the next token**.
- **Self-attention** with queries, keys, values and a **causal mask**.
- **Transformer blocks**: multi-head attention + feed-forward + residuals + LayerNorm.
- **Sampling** with temperature and top-k.

## ✍️ Try it yourself
1. **Hindi / Marathi GPT:** train on a Devanagari text file (see section 2). Does it learn to form valid conjuncts (संयुक्ताक्षर)?
2. Remove the residual connections (`x = self.attn(...)` instead of `x = x + ...`). What happens to training?
3. Replace the character tokenizer with GPT-2's sub-word tokenizer (`pip install tiktoken`, `tiktoken.get_encoding("gpt2")`). What changes?
4. Watch the rest of Karpathy's [**Neural Networks: Zero to Hero**](https://karpathy.ai/zero-to-hero.html) series — especially *micrograd* (how backprop really works) and *Let's build the GPT tokenizer*.
5. Study [nanoGPT](https://github.com/karpathy/nanoGPT) — a clean, fast version of this code that can reproduce GPT-2.

In [ ]:
# Your code here